## Imports

In [ ]:
import random
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from pathlib import Path
from typing import Tuple, List
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.model_selection import StratifiedKFold
import torchvision
print(f"PyTorch version: {torch.__version__}")
print(f"Torchvision version: {torchvision.__version__}")

try:
    from torchvision.models import vit_b_16, ViT_B_16_Weights
    print("\u2713 ViT-B/16 imported from torchvision.models")
except ImportError:
    try:
        import timm
        print("\u2713 Using timm library for ViT")
        HAS_TIMM = True
    except ImportError:
        print("\u26a0 Warning: Neither torchvision ViT nor timm is available")
        print("Installing timm...")
        import subprocess
        subprocess.check_call(['pip', 'install', 'timm'])
        import timm
        HAS_TIMM = True

##CONFIGURATION - ALL HYPERPARAMETERS

In [ ]:
DATA_ROOT = Path("/Users/alimran/Desktop/anthracnose_model_ablation/Dataset_split")
IMG_SIZE = 224
BATCH_SIZE = 16
NUM_WORKERS = 0
EPOCHS = 50
LR = 1e-4
WEIGHT_DECAY = 5e-4
DROPOUT = 0.3
SEED = 250
LOSS_WEIGHT_HEALTH = 1.0
USE_AMP = True
PATIENCE = 5
N_FOLDS = 5

# K-Fold resume state file
STATE_FILE = "vit_base_kfold_state.json"

# Model Config
NUM_SPECIES = 3
NUM_HEALTH = 2
PRETRAINED = False

# Data Augmentation Config
RESIZE_INTERPOLATION = transforms.InterpolationMode.BILINEAR
RANDOM_FLIP_PROB = 0.5
RANDOM_ROTATION_DEGREES = 15
COLOR_JITTER_BRIGHTNESS = 0.2
COLOR_JITTER_CONTRAST = 0.2
COLOR_JITTER_SATURATION = 0.2

# Normalization (ImageNet stats)
NORMALIZE_MEAN = [0.485, 0.456, 0.406]
NORMALIZE_STD = [0.229, 0.224, 0.225]

# Visualization Config
MAX_SAMPLE_IMAGES = 20
CONFUSION_MATRIX_DPI = 150
PLOT_DPI = 150

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Set seeds for reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

## LABEL MAPS

In [ ]:
SPECIES_MAP = {"guava": 0, "mango": 1, "papaya": 2}
HEALTH_MAP = {"healthy": 0, "anthracnose": 1}

def parse_joint_label(folder_name: str) -> Tuple[int, int]:
    """Parse folder name into species and health IDs (case-insensitive)"""
    name = folder_name.strip()
    if "_" not in name:
        raise ValueError(f"Folder name not joint label: {name}")
    sp, he = name.split("_", 1)
    
    # Convert to lowercase for matching
    sp_lower = sp.lower()
    he_lower = he.lower()
    
    if sp_lower not in SPECIES_MAP:
        raise KeyError(f"Unknown species: {sp} (available: {list(SPECIES_MAP.keys())})")
    if he_lower not in HEALTH_MAP:
        raise KeyError(f"Unknown health status: {he} (available: {list(HEALTH_MAP.keys())})")
    
    sp_id = SPECIES_MAP[sp_lower]
    he_id = HEALTH_MAP[he_lower]
    return sp_id, he_id

## DATASET

In [ ]:
class JointLeafDataset(Dataset):
    def __init__(self, split_root: Path, transform=None):
        self.split_root = Path(split_root)
        self.samples: List[Tuple[str, int, int]] = []
        self.transform = transform
        
        # Check if split_root exists
        if not self.split_root.exists():
            raise RuntimeError(
                f"Directory does not exist: {split_root}\n"
                f"Please check if the path is correct."
            )
        
        # Get all subdirectories
        subdirs = [d for d in self.split_root.iterdir() if d.is_dir()]
        
        if len(subdirs) == 0:
            raise RuntimeError(
                f"No subdirectories found in: {split_root}\n"
                f"Expected structure: {split_root}/species_health/images.jpg\n"
                f"Example: {split_root}/guava_healthy/img001.jpg"
            )
        
        print(f"   Found {len(subdirs)} subdirectories in {split_root.name}")
        
        for folder in sorted(subdirs):
            try:
                sp_id, he_id = parse_joint_label(folder.name)
            except (ValueError, KeyError) as e:
                print(f"   ⚠ Skipping folder '{folder.name}': {e}")
                continue
            
            # Find images
            images_in_folder = []
            for p in folder.rglob("*"):
                if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}:
                    images_in_folder.append(str(p))
            
            print(f"   - {folder.name}: {len(images_in_folder)} images")
            
            for img_path in images_in_folder:
                self.samples.append((img_path, sp_id, he_id))
        
        if len(self.samples) == 0:
            raise RuntimeError(
                f"No images found under {split_root}\n"
                f"Found directories: {[d.name for d in subdirs]}\n"
                f"Expected directory names: species_health (e.g., 'guava_healthy' or 'Guava_Healthy')\n"
                f"Supported image formats: .jpg, .jpeg, .png, .bmp\n"
                f"Please check:\n"
                f"  1. Directory names follow 'species_health' format\n"
                f"  2. Images are in correct format\n"
                f"  3. Images are inside the subdirectories"
            )
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        path, sp_id, he_id = self.samples[idx]
        
        try:
            img = Image.open(path).convert('RGB')
        except Exception as e:
            print(f"Error loading image {path}: {e}")
            img = Image.new('RGB', (IMG_SIZE, IMG_SIZE))
        
        if self.transform is not None:
            img = self.transform(img)
        
        return img, torch.tensor(sp_id, dtype=torch.long), torch.tensor(he_id, dtype=torch.long)


class FlatSampleDataset(Dataset):
    """Wraps an explicit list of (path, species_id, health_id) tuples.
    Used to build per-fold train/val loaders without touching the directory structure.
    """
    def __init__(self, samples: List[Tuple[str, int, int]], transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, sp_id, he_id = self.samples[idx]
        try:
            img = Image.open(path).convert('RGB')
        except Exception as e:
            print(f"Error loading image {path}: {e}")
            img = Image.new('RGB', (IMG_SIZE, IMG_SIZE))
        if self.transform is not None:
            img = self.transform(img)
        return img, torch.tensor(sp_id, dtype=torch.long), torch.tensor(he_id, dtype=torch.long)

## TRANSFORMS

In [ ]:
def _pil_to_tensor(img):
    """Convert PIL Image to float tensor [0,1] without using numpy."""
    channels = len(img.getbands())
    data = torch.frombuffer(bytearray(img.tobytes()), dtype=torch.uint8)
    return data.reshape(img.size[1], img.size[0], channels).permute(2, 0, 1).float() / 255.0

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE), interpolation=RESIZE_INTERPOLATION),
    transforms.RandomHorizontalFlip(p=RANDOM_FLIP_PROB),
    transforms.RandomRotation(RANDOM_ROTATION_DEGREES),
    transforms.ColorJitter(
        brightness=COLOR_JITTER_BRIGHTNESS,
        contrast=COLOR_JITTER_CONTRAST,
        saturation=COLOR_JITTER_SATURATION
    ),
    transforms.Lambda(_pil_to_tensor),
    transforms.Normalize(mean=NORMALIZE_MEAN, std=NORMALIZE_STD),
])

eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE), interpolation=RESIZE_INTERPOLATION),
    transforms.Lambda(_pil_to_tensor),
    transforms.Normalize(mean=NORMALIZE_MEAN, std=NORMALIZE_STD),
])

## DATASETS & LOADERS

In [ ]:
print("\n" + "="*80)
print("Loading dataset pool (train + val combined for K-Fold)...")
print("="*80)

# Combine train/ and val/ into a single sample pool for K-Fold splitting
all_samples = []
for split in ["train", "val"]:
    ds = JointLeafDataset(DATA_ROOT / split, transform=None)
    all_samples.extend(ds.samples)
    print()

print(f"\nCombined pool: {len(all_samples)} samples")

# Fixed test set — never included in any fold
print("\nLoading fixed test set...")
test_ds = JointLeafDataset(DATA_ROOT / "test", transform=eval_tf)
test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda")
)
print(f"Test set: {len(test_ds)} images")

# Stratification key: combine species and health into a single class label
strat_labels = [sp * 2 + he for (_, sp, he) in all_samples]

print(f"\n{'='*80}")
print(f"Dataset Summary:")
print(f"  Train+Val pool : {len(all_samples)} images (split per fold)")
print(f"  Test (fixed)   : {len(test_ds)} images")
print(f"  K-Folds        : {N_FOLDS}")
print(f"  Approx per fold - train: {int(len(all_samples)*(N_FOLDS-1)/N_FOLDS)}, "
      f"val: {int(len(all_samples)/N_FOLDS)}")
print("="*80)

## MODEL DEFINITION

In [ ]:
class MultiTaskViTBase(nn.Module):
    def __init__(self, num_species=NUM_SPECIES, num_health=NUM_HEALTH,
                 pretrained=PRETRAINED, dropout=DROPOUT):
        super().__init__()
        
        # Try torchvision first, fallback to timm
        try:
            from torchvision.models import vit_b_16, ViT_B_16_Weights
            if pretrained:
                weights = ViT_B_16_Weights.IMAGENET1K_V1
                self.backbone = vit_b_16(weights=weights)
            else:
                self.backbone = vit_b_16(weights=None)
            
            # Get input dimension from head
            in_dim = self.backbone.heads.head.in_features
            self.backbone.heads.head = nn.Identity()
            print("\u2713 Using ViT-B/16 from torchvision")
            
        except (ImportError, AttributeError):
            import timm
            model_name = 'vit_base_patch16_224.augreg_in21k_ft_in1k' if pretrained else 'vit_base_patch16_224'
            self.backbone = timm.create_model(
                model_name,
                pretrained=pretrained,
                num_classes=0
            )
            in_dim = self.backbone.num_features
            print("\u2713 Using ViT-B/16 from timm library")
        
        self.dropout = nn.Dropout(dropout)
        self.head_species = nn.Linear(in_dim, num_species)
        self.head_health = nn.Linear(in_dim, num_health)
    
    def forward(self, x):
        feats = self.backbone(x)
        feats = self.dropout(feats)
        logits_species = self.head_species(feats)
        logits_health = self.head_health(feats)
        return logits_species, logits_health

print(f"\u2713 Model class defined (instantiated fresh per fold)")

##  OPTIMIZER, SCHEDULER, LOSS FUNCTIONS

In [ ]:
# Shared loss functions — stateless, reused across all folds
criterion_species = nn.CrossEntropyLoss()
criterion_health  = nn.CrossEntropyLoss()
print("\u2713 Loss functions initialized")

## TRAINING UTILITIES

In [ ]:
def accuracy(logits, targets):
    """Calculate accuracy from logits and targets"""
    preds = logits.argmax(dim=1)
    return (preds == targets).float().mean().item()

def run_epoch(loader, model, optimizer=None, train=True, epoch=0):
    """Run one epoch of training or evaluation"""
    if train:
        model.train()
    else:
        model.eval()
    
    running = {
        "loss": 0.0,
        "acc_species": 0.0,
        "acc_health": 0.0,
        "acc_both": 0.0,
        "n": 0
    }
    total_batches = len(loader)
    
    for batch_idx, (imgs, y_species, y_health) in enumerate(loader):
        imgs = imgs.to(device, non_blocking=True)
        y_species = y_species.to(device, non_blocking=True)
        y_health = y_health.to(device, non_blocking=True)
        
        with torch.set_grad_enabled(train):
            if USE_AMP and device.type == "cuda":
                with torch.amp.autocast('cuda'):
                    logits_species, logits_health = model(imgs)
                    loss = criterion_species(logits_species, y_species) + \
                           LOSS_WEIGHT_HEALTH * criterion_health(logits_health, y_health)
            else:
                logits_species, logits_health = model(imgs)
                loss = criterion_species(logits_species, y_species) + \
                       LOSS_WEIGHT_HEALTH * criterion_health(logits_health, y_health)
        
        if train:
            optimizer.zero_grad(set_to_none=True)
            if USE_AMP and device.type == "cuda":
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                optimizer.step()
        
        # Calculate accuracies
        acc_sp = accuracy(logits_species, y_species)
        acc_he = accuracy(logits_health, y_health)
        preds_sp = logits_species.argmax(1)
        preds_he = logits_health.argmax(1)
        acc_both = ((preds_sp == y_species) & (preds_he == y_health)).float().mean().item()
        
        # Update running statistics
        bs = imgs.size(0)
        running["loss"] += loss.item() * bs
        running["acc_species"] += acc_sp * bs
        running["acc_health"] += acc_he * bs
        running["acc_both"] += acc_both * bs
        running["n"] += bs
        
        # Print progress
        if (batch_idx + 1) % max(1, total_batches // 10) == 0 or (batch_idx + 1) == total_batches:
            avg_loss = running["loss"] / running["n"]
            avg_sp = running["acc_species"] / running["n"]
            avg_he = running["acc_health"] / running["n"]
            avg_both = running["acc_both"] / running["n"]
            print(f"  [{batch_idx + 1}/{total_batches}] loss: {avg_loss:.4f}, "
                  f"sp: {avg_sp:.3f}, he: {avg_he:.3f}, both: {avg_both:.3f}")
    
    # Calculate averages
    for k in ["loss", "acc_species", "acc_health", "acc_both"]:
        running[k] /= max(1, running["n"])
    
    return running

## FOLD TRAINING FUNCTION

In [ ]:
def find_latest_epoch_ckpt(fold_idx):
    """Return the Path of the highest-epoch checkpoint for this fold, or None."""
    ckpt_files = list(Path(".").glob(f"fold_{fold_idx}_epoch_*.pt"))
    if not ckpt_files:
        return None
    best, best_ep = None, -1
    for f in ckpt_files:
        try:
            ep = int(f.stem.split("_epoch_")[1])
            if ep > best_ep:
                best_ep, best = ep, f
        except Exception:
            pass
    return best


def train_fold(fold_idx, train_samples, val_samples):
    """
    Train one fold from scratch (or resume from a checkpoint).
    Returns {'species_accuracy': float, 'health_accuracy': float} on the fixed test set.
    """
    global scaler  # run_epoch uses this global

    fold_seed = SEED + fold_idx
    random.seed(fold_seed)
    np.random.seed(fold_seed)
    torch.manual_seed(fold_seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(fold_seed)

    # DataLoaders
    train_loader_fold = DataLoader(
        FlatSampleDataset(train_samples, train_tf),
        batch_size=BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda")
    )
    val_loader_fold = DataLoader(
        FlatSampleDataset(val_samples, eval_tf),
        batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda")
    )

    # Fresh model, optimizer, scheduler, scaler per fold
    model = MultiTaskViTBase(
        num_species=NUM_SPECIES, num_health=NUM_HEALTH,
        pretrained=PRETRAINED, dropout=DROPOUT
    ).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP and device.type == "cuda")

    start_epoch = 0
    best_val_avg = 0.0
    epochs_no_improve = 0
    history = {k: [] for k in [
        "train_loss", "val_loss",
        "train_acc_species", "val_acc_species",
        "train_acc_health", "val_acc_health",
    ]}

    # Resume from latest epoch checkpoint if available
    resume_ckpt = find_latest_epoch_ckpt(fold_idx)
    if resume_ckpt is not None:
        print(f"  Resuming fold {fold_idx+1} from checkpoint: {resume_ckpt}")
        ckpt = torch.load(resume_ckpt, map_location=device)
        model.load_state_dict(ckpt["model"])
        optimizer.load_state_dict(ckpt["optimizer"])
        scheduler.load_state_dict(ckpt["scheduler"])
        start_epoch       = ckpt["epoch"] + 1
        best_val_avg      = ckpt["best_val_avg"]
        epochs_no_improve = ckpt["epochs_no_improve"]
        history           = ckpt["history"]
        print(f"  Resuming from epoch {start_epoch}, best_val_avg={best_val_avg:.4f}")

    # Training loop
    for epoch in range(start_epoch, EPOCHS):
        print(f"\n  Epoch {epoch+1}/{EPOCHS} | LR: {optimizer.param_groups[0]['lr']:.2e}")

        train_stats = run_epoch(train_loader_fold, model, optimizer, train=True,  epoch=epoch)
        val_stats   = run_epoch(val_loader_fold,   model, optimizer=None, train=False, epoch=epoch)
        scheduler.step()

        history["train_loss"].append(train_stats["loss"])
        history["val_loss"].append(val_stats["loss"])
        history["train_acc_species"].append(train_stats["acc_species"])
        history["val_acc_species"].append(val_stats["acc_species"])
        history["train_acc_health"].append(train_stats["acc_health"])
        history["val_acc_health"].append(val_stats["acc_health"])

        print(f"  Train - loss: {train_stats['loss']:.4f} | species: {train_stats['acc_species']:.3f} | health: {train_stats['acc_health']:.3f}")
        print(f"  Val   - loss: {val_stats['loss']:.4f}   | species: {val_stats['acc_species']:.3f} | health: {val_stats['acc_health']:.3f}")

        # Best model criterion: average of species + health val accuracy
        val_avg = (val_stats["acc_species"] + val_stats["acc_health"]) / 2
        if val_avg > best_val_avg:
            best_val_avg = val_avg
            epochs_no_improve = 0
            torch.save({"model": model.state_dict(), "epoch": epoch, "val_avg": best_val_avg},
                       f"fold_{fold_idx}_best.pt")
            print(f"  \u2605 New best saved (val_avg={best_val_avg:.4f})")
        else:
            epochs_no_improve += 1
            print(f"  No improvement ({epochs_no_improve}/{PATIENCE})")

        # Save epoch checkpoint for resume (overwrites previous to save disk)
        prev_ckpt = find_latest_epoch_ckpt(fold_idx)
        torch.save({
            "model":             model.state_dict(),
            "optimizer":         optimizer.state_dict(),
            "scheduler":         scheduler.state_dict(),
            "epoch":             epoch,
            "best_val_avg":      best_val_avg,
            "epochs_no_improve": epochs_no_improve,
            "history":           history,
        }, f"fold_{fold_idx}_epoch_{epoch}.pt")
        if prev_ckpt is not None and prev_ckpt != Path(f"fold_{fold_idx}_epoch_{epoch}.pt"):
            try:
                prev_ckpt.unlink()
            except Exception:
                pass

        if epochs_no_improve >= PATIENCE:
            print(f"  Early stopping triggered after {PATIENCE} epochs without improvement.")
            break

    # Save fold history to CSV
    pd.DataFrame(history).to_csv(f"fold_{fold_idx}_history.csv", index=False)
    print(f"  Saved fold_{fold_idx}_history.csv")

    # Evaluate best checkpoint on fixed test set
    best_ckpt = torch.load(f"fold_{fold_idx}_best.pt", map_location=device)
    model.load_state_dict(best_ckpt["model"])
    model.eval()

    all_sp_preds, all_sp_true = [], []
    all_he_preds, all_he_true = [], []
    with torch.no_grad():
        for imgs, y_sp, y_he in test_loader:
            imgs = imgs.to(device)
            ls, lh = model(imgs)
            all_sp_preds.extend(ls.argmax(1).cpu().numpy())
            all_sp_true.extend(y_sp.numpy())
            all_he_preds.extend(lh.argmax(1).cpu().numpy())
            all_he_true.extend(y_he.numpy())

    sp_acc = accuracy_score(all_sp_true, all_sp_preds)
    he_acc = accuracy_score(all_he_true, all_he_preds)

    print(f"\n  Fold {fold_idx+1} Test Results:")
    print(f"    Species Accuracy : {sp_acc*100:.2f}%")
    print(f"    Health Accuracy  : {he_acc*100:.2f}%")

    return {"species_accuracy": sp_acc, "health_accuracy": he_acc}

## K-FOLD EXECUTION

In [ ]:
def load_kfold_state():
    if Path(STATE_FILE).exists():
        with open(STATE_FILE) as f:
            return json.load(f)
    return {"completed_folds": [], "fold_results": {}}

def save_kfold_state(state):
    with open(STATE_FILE, "w") as f:
        json.dump(state, f, indent=2)


state = load_kfold_state()

if state["completed_folds"]:
    print(f"Resuming K-Fold run. Completed folds: {[f+1 for f in state['completed_folds']]}")
else:
    print("Starting fresh K-Fold run.")

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
fold_splits = list(skf.split(all_samples, strat_labels))

for fold, (train_idx, val_idx) in enumerate(fold_splits):
    if fold in state["completed_folds"]:
        r = state["fold_results"][str(fold)]
        print(f"Fold {fold+1}/{N_FOLDS} — already done "
              f"(species={r['species_accuracy']*100:.2f}%, health={r['health_accuracy']*100:.2f}%)")
        continue

    print(f"\n{'='*70}")
    print(f"FOLD {fold+1}/{N_FOLDS}")
    print(f"  Train: {len(train_idx)} samples | Val: {len(val_idx)} samples")
    print("="*70)

    train_samples_fold = [all_samples[i] for i in train_idx]
    val_samples_fold   = [all_samples[i] for i in val_idx]

    results = train_fold(fold, train_samples_fold, val_samples_fold)

    state["completed_folds"].append(fold)
    state["fold_results"][str(fold)] = results
    save_kfold_state(state)
    print(f"\n\u2713 Fold {fold+1} complete. State saved to {STATE_FILE}")

print("\n" + "="*70)
print("ALL FOLDS COMPLETE")
print("="*70)

## COMPREHENSIVE TESTING FUNCTION

In [ ]:
def comprehensive_test(model, test_loader, device, species_map, health_map):
    """
    Perform comprehensive testing with metrics and visualizations
    """
    model.eval()
    
    # Storage for predictions and ground truth
    all_species_preds = []
    all_species_true = []
    all_health_preds = []
    all_health_true = []
    all_both_correct = []
    
    print("Running comprehensive test evaluation...")
    
    with torch.no_grad():
        for batch_idx, (imgs, y_species, y_health) in enumerate(test_loader):
            imgs = imgs.to(device, non_blocking=True)
            y_species = y_species.to(device, non_blocking=True)
            y_health = y_health.to(device, non_blocking=True)
            
            # Get predictions
            logits_species, logits_health = model(imgs)
            preds_species = logits_species.argmax(dim=1)
            preds_health = logits_health.argmax(dim=1)
            
            # Store predictions and ground truth
            all_species_preds.extend(preds_species.cpu().numpy())
            all_species_true.extend(y_species.cpu().numpy())
            all_health_preds.extend(preds_health.cpu().numpy())
            all_health_true.extend(y_health.cpu().numpy())
            
            # Check if both predictions are correct
            both_correct = ((preds_species == y_species) & (preds_health == y_health)).cpu().numpy()
            all_both_correct.extend(both_correct)
    
    # Convert to numpy arrays
    all_species_preds = np.array(all_species_preds)
    all_species_true = np.array(all_species_true)
    all_health_preds = np.array(all_health_preds)
    all_health_true = np.array(all_health_true)
    all_both_correct = np.array(all_both_correct)
    
    # Reverse mapping for labels
    species_labels = {v: k.capitalize() for k, v in species_map.items()}
    health_labels = {v: k.capitalize() for k, v in health_map.items()}
    
    # -------------------------------
    # Print Metrics
    # -------------------------------
    print("\n" + "="*80)
    print("COMPREHENSIVE TEST RESULTS")
    print("="*80)
    
    # Overall accuracies
    species_acc = accuracy_score(all_species_true, all_species_preds)
    health_acc = accuracy_score(all_health_true, all_health_preds)
    both_acc = all_both_correct.mean()
    
    print(f"\n{'OVERALL ACCURACIES':^80}")
    print("-"*80)
    print(f"  Species Classification:  {species_acc:.4f} ({species_acc*100:.2f}%)")
    print(f"  Health Classification:   {health_acc:.4f} ({health_acc*100:.2f}%)")
    print(f"  Both Correct (Joint):    {both_acc:.4f} ({both_acc*100:.2f}%)")
    
    # Species Classification Report
    print(f"\n{'SPECIES CLASSIFICATION REPORT':^80}")
    print("-"*80)
    print(classification_report(
        all_species_true,
        all_species_preds,
        target_names=[species_labels[i] for i in sorted(species_labels.keys())],
        digits=4
    ))
    
    # Health Classification Report
    print(f"\n{'HEALTH/DISEASE CLASSIFICATION REPORT':^80}")
    print("-"*80)
    print(classification_report(
        all_health_true,
        all_health_preds,
        target_names=[health_labels[i] for i in sorted(health_labels.keys())],
        digits=4
    ))
    
    # Per-class joint accuracy
    print(f"\n{'PER-CLASS JOINT ACCURACY':^80}")
    print("-"*80)
    for sp_id in sorted(species_labels.keys()):
        for he_id in sorted(health_labels.keys()):
            # Find samples of this joint class
            mask = (all_species_true == sp_id) & (all_health_true == he_id)
            if mask.sum() > 0:
                joint_acc = all_both_correct[mask].mean()
                count = mask.sum()
                sp_name = species_labels[sp_id]
                he_name = health_labels[he_id]
                print(f"  {sp_name:8s} + {he_name:12s}: {joint_acc:.4f} "
                      f"({joint_acc*100:.2f}%) [{count:4d} samples]")
    
    # -------------------------------
    # Visualizations
    # -------------------------------
    
    # 1. Species Confusion Matrix
    fig, ax = plt.subplots(figsize=(8, 6))
    cm_species = confusion_matrix(all_species_true, all_species_preds)
    sns.heatmap(
        cm_species,
        annot=True,
        fmt='d',
        cmap='Blues',
        ax=ax,
        xticklabels=[species_labels[i] for i in sorted(species_labels.keys())],
        yticklabels=[species_labels[i] for i in sorted(species_labels.keys())]
    )
    ax.set_title(f'Species Classification\nAccuracy: {species_acc:.2%}',
                 fontsize=12, fontweight='bold')
    ax.set_ylabel('True Label')
    ax.set_xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig('confusion_matrix_species.png', dpi=CONFUSION_MATRIX_DPI, bbox_inches='tight')
    print(f"\nSaved species confusion matrix to 'confusion_matrix_species.png'")
    plt.close()
    
    # 2. Health Confusion Matrix
    fig, ax = plt.subplots(figsize=(8, 6))
    cm_health = confusion_matrix(all_health_true, all_health_preds)
    sns.heatmap(
        cm_health,
        annot=True,
        fmt='d',
        cmap='Greens',
        ax=ax,
        xticklabels=[health_labels[i] for i in sorted(health_labels.keys())],
        yticklabels=[health_labels[i] for i in sorted(health_labels.keys())]
    )
    ax.set_title(f'Health/Disease Classification\nAccuracy: {health_acc:.2%}',
                 fontsize=12, fontweight='bold')
    ax.set_ylabel('True Label')
    ax.set_xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig('confusion_matrix_health.png', dpi=CONFUSION_MATRIX_DPI, bbox_inches='tight')
    print(f"Saved health confusion matrix to 'confusion_matrix_health.png'")
    plt.close()
    
    # 3. Joint Accuracy Heatmap
    fig, ax = plt.subplots(figsize=(8, 6))
    joint_acc_matrix = np.zeros((len(species_labels), len(health_labels)))
    
    for sp_id in sorted(species_labels.keys()):
        for he_id in sorted(health_labels.keys()):
            mask = (all_species_true == sp_id) & (all_health_true == he_id)
            if mask.sum() > 0:
                joint_acc_matrix[sp_id, he_id] = all_both_correct[mask].mean()
    
    sns.heatmap(
        joint_acc_matrix,
        annot=True,
        fmt='.3f',
        cmap='RdYlGn',
        xticklabels=[health_labels[i] for i in sorted(health_labels.keys())],
        yticklabels=[species_labels[i] for i in sorted(species_labels.keys())],
        vmin=0,
        vmax=1,
        ax=ax,
        cbar_kws={'label': 'Accuracy'}
    )
    ax.set_title('Joint Classification Accuracy by Class', fontsize=12, fontweight='bold')
    ax.set_ylabel('Species')
    ax.set_xlabel('Health Status')
    plt.tight_layout()
    plt.savefig('joint_accuracy_heatmap.png', dpi=CONFUSION_MATRIX_DPI, bbox_inches='tight')
    print(f"Saved joint accuracy heatmap to 'joint_accuracy_heatmap.png'")
    plt.close()
    
    print("\n" + "="*80)
    print("Testing complete! Generated visualizations:")
    print("  - confusion_matrix_species.png")
    print("  - confusion_matrix_health.png")
    print("  - joint_accuracy_heatmap.png")
    print("="*80 + "\n")
    
    return {
        'species_accuracy': species_acc,
        'health_accuracy': health_acc,
        'joint_accuracy': both_acc,
        'species_preds': all_species_preds,
        'species_true': all_species_true,
        'health_preds': all_health_preds,
        'health_true': all_health_true
    }

## BEST FOLD EVALUATION

In [ ]:
# Identify best fold by health accuracy
state = load_kfold_state()
fold_results_list = [state["fold_results"][str(i)] for i in range(N_FOLDS)]

best_fold_idx = max(range(N_FOLDS),
                    key=lambda i: fold_results_list[i]["health_accuracy"])
print(f"Best fold: Fold {best_fold_idx+1} "
      f"(health_acc={fold_results_list[best_fold_idx]['health_accuracy']*100:.2f}%)")

best_model = MultiTaskViTBase(
    num_species=NUM_SPECIES, num_health=NUM_HEALTH,
    pretrained=PRETRAINED, dropout=DROPOUT
).to(device)
best_ckpt = torch.load(f"fold_{best_fold_idx}_best.pt", map_location=device)
best_model.load_state_dict(best_ckpt["model"])
print(f"Loaded best fold checkpoint (epoch {best_ckpt['epoch']+1})")

test_results = comprehensive_test(
    model=best_model,
    test_loader=test_loader,
    device=device,
    species_map=SPECIES_MAP,
    health_map=HEALTH_MAP
)

## SAMPLE PREDICTIONS (BEST FOLD)

In [ ]:
print("\n" + "="*80)
print("Generating Sample Predictions Visualization (Best Fold)")
print("="*80 + "\n")

# Rebuild val set for the best fold
skf_temp = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
fold_splits_temp = list(skf_temp.split(all_samples, strat_labels))
_, val_idx_best = fold_splits_temp[best_fold_idx]
val_samples_best = [all_samples[i] for i in val_idx_best]
val_loader_best = DataLoader(
    FlatSampleDataset(val_samples_best, eval_tf),
    batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda")
)

species_labels_map = {0: 'Guava', 1: 'Mango', 2: 'Papaya'}
health_labels_map  = {0: 'Healthy', 1: 'Anthracnose'}
amount = 10

sample_images_by_class = {0: [], 1: [], 2: []}
sample_preds_by_class   = {0: [], 1: [], 2: []}
sample_gt_by_class      = {0: [], 1: [], 2: []}

best_model.eval()
with torch.no_grad():
    for images, species_batch, health_batch in val_loader_best:
        images = images.to(device)
        sp_preds = best_model(images)[0].argmax(1)
        he_preds = best_model(images)[1].argmax(1)
        for i in range(len(images)):
            sc = species_batch[i].item()
            if len(sample_images_by_class[sc]) < amount:
                sample_images_by_class[sc].append(images[i].cpu())
                sample_preds_by_class[sc].append(
                    {'species': sp_preds[i].item(), 'health': he_preds[i].item()})
                sample_gt_by_class[sc].append(
                    {'species': species_batch[i].item(), 'health': health_batch[i].item()})
        if all(len(v) >= amount for v in sample_images_by_class.values()):
            break

mean_arr = np.array(NORMALIZE_MEAN)
std_arr  = np.array(NORMALIZE_STD)
fig, axes = plt.subplots(NUM_SPECIES, amount, figsize=(3*amount, 9))
for row, sp_idx in enumerate(sorted(sample_images_by_class.keys())):
    for col in range(amount):
        ax = axes[row, col]
        img  = sample_images_by_class[sp_idx][col]
        pred = sample_preds_by_class[sp_idx][col]
        gt   = sample_gt_by_class[sp_idx][col]
        img_d = np.clip(std_arr * img.numpy().transpose(1,2,0) + mean_arr, 0, 1)
        ax.imshow(img_d)
        ax.axis('off')
        both_ok = (pred['species'] == gt['species']) and (pred['health'] == gt['health'])
        title = (f"Pred: {species_labels_map[pred['species']]}, {health_labels_map[pred['health']]}\n"
                 f"True: {species_labels_map[gt['species']]}, {health_labels_map[gt['health']]}")
        ax.set_title(title, fontsize=8, color='green' if both_ok else 'red', fontweight='bold')
    fig.text(0.02, 0.5 + (1 - row) * 0.3, species_labels_map[sp_idx],
             fontsize=12, fontweight='bold', va='center', rotation=90)

plt.suptitle(f'Sample Predictions - Best Fold {best_fold_idx+1} ({amount} per class)\n(Green=Correct, Red=Incorrect)',
             fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout(rect=[0.05, 0, 1, 0.99])
plt.savefig('sample_predictions.png', dpi=CONFUSION_MATRIX_DPI, bbox_inches='tight')
print(f"Saved sample_predictions.png")
plt.close()

## FINAL K-FOLD RESULTS (Mean ± Std)

In [ ]:
state = load_kfold_state()
fold_results_list = [state["fold_results"][str(i)] for i in range(N_FOLDS)]

sp_vals = [r["species_accuracy"] * 100 for r in fold_results_list]
he_vals = [r["health_accuracy"]  * 100 for r in fold_results_list]

print("\n" + "="*70)
print("K-FOLD CROSS-VALIDATION RESULTS")
print("="*70)

print(f"\n{'Fold':<8} {'Species Acc (%)':>16} {'Health Acc (%)':>16}")
print("-"*42)
for i, r in enumerate(fold_results_list):
    print(f"Fold {i+1:<3} {r['species_accuracy']*100:>16.2f} {r['health_accuracy']*100:>16.2f}")
print("-"*42)
print(f"{'Mean':<8} {np.mean(sp_vals):>16.2f} {np.mean(he_vals):>16.2f}")
print(f"{'Std':<8} {np.std(sp_vals):>16.2f} {np.std(he_vals):>16.2f}")

print("\n" + "="*70)
print("SUMMARY  (mean \u00b1 std)")
print("="*70)
print(f"  Species Accuracy : {np.mean(sp_vals):.2f} \u00b1 {np.std(sp_vals):.2f}")
print(f"  Health Accuracy  : {np.mean(he_vals):.2f} \u00b1 {np.std(he_vals):.2f}")
print("="*70)
print(f"\nConfig: N_FOLDS={N_FOLDS}, BATCH_SIZE={BATCH_SIZE}, LR={LR}, "
      f"WEIGHT_DECAY={WEIGHT_DECAY}, DROPOUT={DROPOUT}, EPOCHS={EPOCHS}, PATIENCE={PATIENCE}")

## END

In [ ]:
print("\n" + "="*70)
print("ALL TASKS COMPLETED")
print("="*70)
print("Generated files:")
print(f"  {STATE_FILE:<35} K-Fold resume state")
for i in range(N_FOLDS):
    print(f"  fold_{i}_best.pt{' '*19} Best model — fold {i+1}")
    print(f"  fold_{i}_history.csv{' '*15} Training history — fold {i+1}")
print("  confusion_matrix_species.png       Best fold species confusion matrix")
print("  confusion_matrix_health.png        Best fold health confusion matrix")
print("  joint_accuracy_heatmap.png         Best fold joint accuracy heatmap")
print("  sample_predictions.png             Best fold sample predictions")